In [25]:
import os
import cv2
import random
import numpy as np
from glob import glob

# -----------------------
# 路径
normal_dir = "D:/Baggage_XRay/train/Normal"
patch_dir  = "D:/Baggage_XRay/selected_patches"
save_dir   = "D:/Baggage_XRay/synthetic/Abnormal"
mask_dir   = "D:/Baggage_XRay/synthetic/Mask"

os.makedirs(save_dir, exist_ok=True)
os.makedirs(mask_dir, exist_ok=True)

# -----------------------
# 读取图片
exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
normal_images = []
patch_images  = []
for e in exts:
    normal_images.extend(glob(os.path.join(normal_dir, e)))
    patch_images.extend(glob(os.path.join(patch_dir, e)))

print("找到 Normal:", len(normal_images))
print("找到 Patch :", len(patch_images))


# -----------------------
# 粘贴 patch 到 Normal 图物体内部（不缩小）
def paste_patch_on_object_no_resize(normal_img, patch_img):
    h, w, _ = normal_img.shape

    # patch mask（黑背景=0）
    gray_patch = cv2.cvtColor(patch_img, cv2.COLOR_BGR2GRAY)
    _, patch_mask = cv2.threshold(gray_patch, 10, 255, cv2.THRESH_BINARY)
    ph, pw = patch_mask.shape

    # 获取 Normal 图物体 bboxes
    bboxes = get_object_bboxes(normal_img)
    if not bboxes:
        # 没找到物体 → 随机放
        x2 = random.randint(0, max(0, w - pw))
        y2 = random.randint(0, max(0, h - ph))
    else:
        # 随机选择一个物体 bbox
        x_obj, y_obj, obj_w, obj_h = random.choice(bboxes)

        # patch 尺寸比物体大 → 裁剪 patch，使其能放入物体
        if pw > obj_w:
            patch_img = patch_img[:, :obj_w]
            patch_mask = patch_mask[:, :obj_w]
            pw = obj_w
        if ph > obj_h:
            patch_img = patch_img[:obj_h, :]
            patch_mask = patch_mask[:obj_h, :]
            ph = obj_h

        # 在物体 bbox 内随机选择位置
        x_min = x_obj
        x_max = x_obj + obj_w - pw
        y_min = y_obj
        y_max = y_obj + obj_h - ph
        if x_max <= x_min or y_max <= y_min:
            x2 = x_obj
            y2 = y_obj
        else:
            x2 = random.randint(x_min, x_max)
            y2 = random.randint(y_min, y_max)

    # 粘贴 patch
    img_copy = normal_img.copy()
    roi = img_copy[y2:y2+ph, x2:x2+pw]
    mask_inv = cv2.bitwise_not(patch_mask)
    bg = cv2.bitwise_and(roi, roi, mask=mask_inv)
    fg = cv2.bitwise_and(patch_img, patch_img, mask=patch_mask)
    img_copy[y2:y2+ph, x2:x2+pw] = cv2.add(bg, fg)

    # mask 记录 patch 位置
    mask_out = np.zeros((h, w), dtype=np.uint8)
    mask_out[y2:y2+ph, x2:x2+pw] = patch_mask

    return img_copy, mask_out


# -----------------------
# 生成 synthetic abnormal（每张 Normal 图贴 1 个 patch）
for idx, normal_path in enumerate(normal_images[:30]):  # 测试前 20 张
    normal_img = cv2.imread(normal_path)
    if normal_img is None:
        print(f"❌ 读取失败: {normal_path}")
        continue

    patch_img = cv2.imread(random.choice(patch_images))
    if patch_img is None:
        print("❌ patch 读取失败")
        continue

    synth_img, mask = paste_patch_on_object(normal_img, patch_img)

    # 保存
    cv2.imwrite(os.path.join(save_dir, f"synthetic_{idx}.jpg"), synth_img)
    cv2.imwrite(os.path.join(mask_dir, f"synthetic_{idx}.png"), mask)

    print(f"✅ 已生成 synthetic_{idx}.jpg")

print("🎉 Synthetic abnormal data generated!")


找到 Normal: 3296
找到 Patch : 10
✅ 已生成 synthetic_0.jpg
✅ 已生成 synthetic_1.jpg
✅ 已生成 synthetic_2.jpg
✅ 已生成 synthetic_3.jpg
✅ 已生成 synthetic_4.jpg
✅ 已生成 synthetic_5.jpg
✅ 已生成 synthetic_6.jpg
✅ 已生成 synthetic_7.jpg
✅ 已生成 synthetic_8.jpg
✅ 已生成 synthetic_9.jpg
✅ 已生成 synthetic_10.jpg
✅ 已生成 synthetic_11.jpg
✅ 已生成 synthetic_12.jpg
✅ 已生成 synthetic_13.jpg
✅ 已生成 synthetic_14.jpg
✅ 已生成 synthetic_15.jpg
✅ 已生成 synthetic_16.jpg
✅ 已生成 synthetic_17.jpg
✅ 已生成 synthetic_18.jpg
✅ 已生成 synthetic_19.jpg
✅ 已生成 synthetic_20.jpg
✅ 已生成 synthetic_21.jpg
✅ 已生成 synthetic_22.jpg
✅ 已生成 synthetic_23.jpg
✅ 已生成 synthetic_24.jpg
✅ 已生成 synthetic_25.jpg
✅ 已生成 synthetic_26.jpg
✅ 已生成 synthetic_27.jpg
✅ 已生成 synthetic_28.jpg
✅ 已生成 synthetic_29.jpg
🎉 Synthetic abnormal data generated!


In [14]:
import os
import cv2
import random
import numpy as np
from glob import glob

# -----------------------
# 路径
normal_dir = "D:/Baggage_XRay/train/Normal"
patch_dir  = "D:/Baggage_XRay/selected_patches"
save_dir   = "D:/Baggage_XRay/synthetic/Abnormal"
mask_dir   = "D:/Baggage_XRay/synthetic/Mask"

os.makedirs(save_dir, exist_ok=True)
os.makedirs(mask_dir, exist_ok=True)

# -----------------------
# 读取图片
exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
normal_images = []
patch_images  = []
for e in exts:
    normal_images.extend(glob(os.path.join(normal_dir, e)))
    patch_images.extend(glob(os.path.join(patch_dir, e)))

print("找到 Normal:", len(normal_images))
print("找到 Patch :", len(patch_images))


# -----------------------
# 获取 Normal 图中的物体 bboxes（去掉最大的 tray）
def get_object_bboxes(normal_img):
    gray = cv2.cvtColor(normal_img, cv2.COLOR_BGR2GRAY)
    _, th = cv2.threshold(gray, 20, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []

    # 按面积排序，去掉最大的（tray）
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    if len(contours) > 1:
        selected = contours[1:]  # tray 之后的 object
    else:
        selected = contours
    bboxes = [cv2.boundingRect(c) for c in selected]
    return bboxes


# -----------------------
# 粘贴 patch 到 Normal 图的 object 内（不缩小）
def paste_patch_on_object_no_resize(normal_img, patch_img):
    h, w, _ = normal_img.shape

    # patch mask（黑背景=0 → 物体=255）
    gray_patch = cv2.cvtColor(patch_img, cv2.COLOR_BGR2GRAY)
    _, patch_mask = cv2.threshold(gray_patch, 10, 255, cv2.THRESH_BINARY)
    ph, pw = patch_mask.shape

    # 获取 object bboxes（不包含 tray）
    bboxes = get_object_bboxes(normal_img)
    if not bboxes:
        # 没找到 object → 随机放
        x2 = random.randint(0, max(0, w - pw))
        y2 = random.randint(0, max(0, h - ph))
    else:
        # 随机选择一个 object bbox
        x_obj, y_obj, obj_w, obj_h = random.choice(bboxes)

        # patch 比 object 大 → 裁剪 patch
        if pw > obj_w:
            patch_img = patch_img[:, :obj_w]
            patch_mask = patch_mask[:, :obj_w]
            pw = obj_w
        if ph > obj_h:
            patch_img = patch_img[:obj_h, :]
            patch_mask = patch_mask[:obj_h, :]
            ph = obj_h

        # 保证 patch 在 object 内
        x_min = x_obj
        x_max = x_obj + obj_w - pw
        y_min = y_obj
        y_max = y_obj + obj_h - ph
        if x_max <= x_min or y_max <= y_min:
            x2 = x_obj
            y2 = y_obj
        else:
            x2 = random.randint(x_min, x_max)
            y2 = random.randint(y_min, y_max)

    # 粘贴 patch
    img_copy = normal_img.copy()
    roi = img_copy[y2:y2+ph, x2:x2+pw]
    mask_inv = cv2.bitwise_not(patch_mask)
    bg = cv2.bitwise_and(roi, roi, mask=mask_inv)
    fg = cv2.bitwise_and(patch_img, patch_img, mask=patch_mask)
    img_copy[y2:y2+ph, x2:x2+pw] = cv2.add(bg, fg)

    # 输出 mask
    mask_out = np.zeros((h, w), dtype=np.uint8)
    mask_out[y2:y2+ph, x2:x2+pw] = patch_mask

    return img_copy, mask_out


# -----------------------
# 生成 synthetic abnormal（每张 Normal 图贴 1 个 patch）
for idx, normal_path in enumerate(normal_images[:30]):  # 测试 30 张
    normal_img = cv2.imread(normal_path)
    if normal_img is None:
        print(f"❌ 读取失败: {normal_path}")
        continue

    patch_img = cv2.imread(random.choice(patch_images))
    if patch_img is None:
        print("❌ patch 读取失败")
        continue

    synth_img, mask = paste_patch_on_object_no_resize(normal_img, patch_img)

    # 保存
    cv2.imwrite(os.path.join(save_dir, f"synthetic_{idx}.jpg"), synth_img)
    cv2.imwrite(os.path.join(mask_dir, f"synthetic_{idx}.png"), mask)

    print(f"✅ 已生成 synthetic_{idx}.jpg")

print("🎉 Synthetic abnormal data generated!")


找到 Normal: 3296
找到 Patch : 10
✅ 已生成 synthetic_0.jpg
✅ 已生成 synthetic_1.jpg
✅ 已生成 synthetic_2.jpg
✅ 已生成 synthetic_3.jpg
✅ 已生成 synthetic_4.jpg
✅ 已生成 synthetic_5.jpg
✅ 已生成 synthetic_6.jpg
✅ 已生成 synthetic_7.jpg
✅ 已生成 synthetic_8.jpg
✅ 已生成 synthetic_9.jpg
✅ 已生成 synthetic_10.jpg
✅ 已生成 synthetic_11.jpg
✅ 已生成 synthetic_12.jpg
✅ 已生成 synthetic_13.jpg
✅ 已生成 synthetic_14.jpg
✅ 已生成 synthetic_15.jpg
✅ 已生成 synthetic_16.jpg
✅ 已生成 synthetic_17.jpg
✅ 已生成 synthetic_18.jpg
✅ 已生成 synthetic_19.jpg
✅ 已生成 synthetic_20.jpg
✅ 已生成 synthetic_21.jpg
✅ 已生成 synthetic_22.jpg
✅ 已生成 synthetic_23.jpg
✅ 已生成 synthetic_24.jpg
✅ 已生成 synthetic_25.jpg
✅ 已生成 synthetic_26.jpg
✅ 已生成 synthetic_27.jpg
✅ 已生成 synthetic_28.jpg
✅ 已生成 synthetic_29.jpg
🎉 Synthetic abnormal data generated!


In [ ]:
import albumentations as A
import cv2
import os

# 只做随机旋转
transform = A.Compose([
    A.RandomRotate90(p=1.0),  # 随机 0°, 90°, 180°, 270°
    A.Rotate(limit=30, p=1.0) # ±30° 随机旋转
])

# 输入图片路径
input_image_path = r"D:/Baggage_XRay/test/Abnormal/1 (1).png"

# 输出文件夹
output_dir = r"D:/Baggage_XRay/albumemtations"
os.makedirs(output_dir, exist_ok=True)

# 读取图片
image = cv2.imread(input_image_path)
if image is None:
    raise ValueError(f"❌ 无法读取图片: {input_image_path}")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 生成并保存 10 张增强图片
for i in range(10):
    augmented = transform(image=image)["image"]
    save_path = os.path.join(output_dir, f"aug_{i}.png")  # 保存为png
    cv2.imwrite(save_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))

print(f"✅ 10 张随机旋转图片已保存到: {output_dir}")


✅ 10 张随机旋转图片已保存到: D:/Baggage_XRay/albumemtations


In [25]:
import albumentations as A
import cv2
import os

# 定义增强 pipeline
transform = A.Compose([
    A.HorizontalFlip(p=1.0),   # 总是水平翻转
    A.RandomRotate90(p=1.0)    # 随机旋转 0°, 90°, 180°, 270°
])

# 输入图片路径
input_image_path = r"D:/Baggage_XRay/test/Abnormal/1 (1).png"

# 输出文件夹
output_dir = r"D:/Baggage_XRay/albumemtations"
os.makedirs(output_dir, exist_ok=True)

# 读取图片
image = cv2.imread(input_image_path)
if image is None:
    raise ValueError(f"❌ 无法读取图片: {input_image_path}")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 生成并保存 10 张增强图片
for i in range(10):
    augmented = transform(image=image)["image"]
    save_path = os.path.join(output_dir, f"aug_{i}.png")
    cv2.imwrite(save_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))

print(f"✅ 10 张水平翻转 + 随机旋转图片已保存到: {output_dir}")


✅ 10 张水平翻转 + 随机旋转图片已保存到: D:/Baggage_XRay/albumemtations


In [52]:
import albumentations as A
import cv2
import os
from glob import glob
import math

# 输入输出路径
input_dir = r"D:/Baggage_XRay/train/Abnormal"
output_dir = r"D:/Baggage_XRay/augmented_Abnormal"
os.makedirs(output_dir, exist_ok=True)

# 找到所有图片
image_paths = glob(os.path.join(input_dir, "*.[jJ][pP][gG]")) + \
              glob(os.path.join(input_dir, "*.[pP][nN][gG]"))

num_original = len(image_paths)
target_num = 20
augmentation_factor = math.ceil(target_num / num_original)

print(f"原始图片数量: {num_original}")
print(f"每张图生成: {augmentation_factor} 张增强图")

# 定义增强 pipeline，Affine 保留整张图
transform = A.Compose([
    A.HorizontalFlip(p=1),
    A.VerticalFlip(p=1),
    A.Rotate(limit=15, p=1),
    A.Affine(fit_output=True,fill=0,p=1.0)

])

# 遍历图片
idx = 0
for img_path in image_paths:
    filename = os.path.splitext(os.path.basename(img_path))[0]

    image = cv2.imread(img_path)
    if image is None:
        print(f"❌ 无法读取图片: {img_path}")
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    for i in range(augmentation_factor):
        augmented = transform(image=image)["image"]
        save_path = os.path.join(output_dir, f"{filename}_aug_{i}.jpg")
        cv2.imwrite(save_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))
        idx += 1
        if idx >= target_num:
            break
    if idx >= target_num:
        break

print(f"✅ 增强完成，总共生成 {idx} 张图片，保存到 {output_dir}")


原始图片数量: 74
每张图生成: 1 张增强图
✅ 增强完成，总共生成 20 张图片，保存到 D:/Baggage_XRay/augmented_Abnormal


In [ ]:
import albumentations as A
import cv2
import os
from glob import glob

# 输入输出路径
input_dir = r"D:/Baggage_XRay/train/normal"
output_dir = r"D:/Baggage_XRay/augmented_Normal"
os.makedirs(output_dir, exist_ok=True)

# 找到所有图片
image_paths = glob(os.path.join(input_dir, "*.[jJ][pP][gG]")) + \
              glob(os.path.join(input_dir, "*.[pP][nN][gG]"))

num_original = len(image_paths)
augmentation_factor = 6  # 每张图片生成 6 张增强图

print(f"原始图片数量: {num_original}")
print(f"每张图生成: {augmentation_factor} 张增强图")

# 定义增强 pipeline
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=15, p=1.0, border_mode=cv2.BORDER_REFLECT_101),
    A.Affine(scale=(0.9,1.1), translate_percent=(0,0.05),
             rotate=(-15,15), shear=(-5,5),
             fit_output=True, mode=cv2.BORDER_REFLECT_101, p=0.7),
    A.HueSaturationValue(p=0.5),
])

# 遍历图片
idx = 0
for img_path in image_paths:
    filename = os.path.splitext(os.path.basename(img_path))[0]

    image = cv2.imread(img_path)
    if image is None:
        print(f"❌ 无法读取图片: {img_path}")
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    for i in range(augmentation_factor):
        augmented = transform(image=image)["image"]
        save_path = os.path.join(output_dir, f"{filename}_aug_{i}.jpg")
        cv2.imwrite(save_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))
        idx += 1

print(f"✅ 增强完成，总共生成 {idx} 张图片，保存到 {output_dir}")


原始图片数量: 2354
每张图生成: 6 张增强图


C:\Users\AI Training\AppData\Local\Temp\ipykernel_21148\3566940566.py:26: UserWarning: Argument(s) 'mode' are not valid for transform Affine
  A.Affine(scale=(0.9,1.1), translate_percent=(0,0.05),


✅ 增强完成，总共生成 14124 张图片，保存到 D:/Baggage_XRay/augmented_Normal
